# Tối ưu hoá log-loss — Santander Product Recommendation
### 4 thuật toán: GD · SGD · Subgradient · Proximal

---

## 1. Đặt bài toán

Với mỗi tháng $t$, khách hàng $i$, sản phẩm $k$, nhãn **"thêm mới"**:

$$y_{i,k}(t) = 1 \iff \text{khách } i \text{ có sản phẩm } k \text{ ở tháng } t
\text{ nhưng KHÔNG có ở tháng } t-1$$

Mô hình hồi quy logistic $p_i = \sigma(x_i^\top w)$, $\sigma(z)=1/(1+e^{-z})$.
Hàm mục tiêu là **log-loss** cộng thêm chính quy hoá:

$$\min_{w}\; F(w) = \underbrace{\frac1n\sum_{i=1}^n \Big[\log\big(1+e^{x_i^\top w}\big) - y_i\, x_i^\top w\Big] + \frac{\lambda_2}{2}\|w\|_2^2}_{f(w)\ :\ \textbf{trơn}} \;+\; \underbrace{\lambda_1\|w\|_1}_{g(w)\ :\ \textbf{KHÔNG trơn}}$$

| Thành phần | Tính chất | Thuật toán dùng được |
|---|---|---|
| $f$ = log-loss + L2 | lồi, khả vi, $\nabla f$ Lipschitz với $L=\frac{\sigma_{\max}(X)^2}{4n}+\lambda_2$ | **GD**, **SGD** |
| $g=\lambda_1\|w\|_1$ | lồi, **không khả vi tại 0** | **Subgradient**, **Proximal** |

Dùng log-loss vì nó là negative log-likelihood của mô hình Bernoulli: lồi theo $w$
(mọi cực tiểu địa phương đều là toàn cục) và cho ra xác suất, cần để xếp hạng gợi ý.

### Kế hoạch
1. **Vòng 1** — cả 4 thuật toán dùng **chung một bộ tham số** → kết quả
2. **Vòng 2** — mỗi thuật toán **dò tham số riêng** (cùng ngân sách) → kết quả
3. **Nhận xét**

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from logloss_optim import (sigmoid, logloss, grad, objective, subgradient,
                           prox_l1, prox_elastic_net, lipschitz, kiem_tra_grad,
                           du_doan, gradient_descent, sgd, subgradient_method,
                           proximal_gradient)
from santander_data import build_dataset, TARGET_COLS
from run_optim import (TEN_4, MAU, mo_ta_features, danh_gia, in_bang, do_tham_so,
                       auc, map_at_7, ve_hoi_tu, ve_val, ve_do_thua, ve_2_vong)

np.set_printoptions(precision=4, suppress=True)
print("OK")

## 2. Dữ liệu và bộ đặc trưng

Lần chạy đầu quét `train_ver2.csv` (2.3 GB) lấy 4 tháng cần thiết rồi lưu cache
(~40 giây), các lần sau chỉ mất vài giây.

- **Train**: đặc trưng tháng 2015-05 → nhãn "thêm mới" tháng 2015-06
- **Validation**: đặc trưng tháng 2016-04 → nhãn "thêm mới" tháng 2016-05

In [ ]:
du_lieu = build_dataset()
X_tr = du_lieu["X_tr"].astype(float)       # cot 0 la intercept
Y_tr = du_lieu["Y_tr"]
X_va = du_lieu["X_va"].astype(float)
Y_va = du_lieu["Y_va"]
ten_cot = du_lieu["feature_names"]

SAN_PHAM = "ind_recibo_ult1"               # san pham co ti le them moi cao nhat
k = TARGET_COLS.index(SAN_PHAM)
y_tr = Y_tr[:, k].astype(float)
y_va = Y_va[:, k].astype(float)
n, d = X_tr.shape
val = (X_va, y_va)

mo_ta_features(ten_cot, X_tr, X_va, y_tr, y_va)

In [ ]:
print("Ti le 'them moi' theo san pham (train) - bai toan rat mat can bang:")
for j in np.argsort(-Y_tr.mean(axis=0))[:8]:
    print("  %-22s %6.3f%%" % (TARGET_COLS[j], Y_tr[:, j].mean() * 100))

print("\nMot vai ten cot:")
print("  dau :", ten_cot[:8])
print("  lag :", ten_cot[-24:-21], "...")

## 3. Gradient và hằng số Lipschitz

$$\nabla f(w) = \frac1n X^\top\big(\sigma(Xw) - y\big) + \lambda_2 w$$

Hai việc phải làm **trước khi** chạy bất kỳ thuật toán nào:

1. **Kiểm tra gradient bằng sai phân hữu hạn** — sai số phải rất nhỏ (cỡ $10^{-8}$).
2. **Ước lượng $L$** để lấy bước an toàn $t=1/L$. Vì $\sigma'(z)\le\frac14$ nên
   $L=\frac{\sigma_{\max}(X)^2}{4n}+\lambda_2$, tìm $\sigma_{\max}$ bằng power iteration.

In [ ]:
lam1, lam2 = 1e-3, 1e-6
VONG, B = 60, 1024              # ngan sach chung: 60 lan duyet du lieu

w_thu = np.random.RandomState(0).randn(d) * 0.05
print("sai so gradient (sai phan huu han):", kiem_tra_grad(w_thu, X_tr, y_tr, lam2))

L = lipschitz(X_tr, lam2)
eta = 1 / L
print("L = %.4f  ->  buoc an toan 1/L = %.4f" % (L, eta))

p0 = y_tr.mean()
moc = -(p0 * np.log(p0) + (1 - p0) * np.log(1 - p0))
print("log-loss cua mo hinh hang so = %.6f   <-- moc can vuot" % moc)

## 4. Bốn thuật toán

**4.1 GD** — $w_{k+1} = w_k - t\,\nabla f(w_k)$. Bước $t=1/L$ đảm bảo giảm đơn điệu,
tốc độ $O(1/k)$. Biến thể: **line search Armijo** — thử bước lớn rồi co dần đến khi
$f(w-t\nabla f)\le f(w)-c\,t\|\nabla f\|^2$.

**4.2 SGD** — $w_{k+1} = w_k - \eta_k\nabla f_B(w_k)$ với $B$ là mini-batch ngẫu nhiên.
$\nabla f_B$ là ước lượng **không chệch** của $\nabla f$, rẻ hơn $n/|B|$ lần mỗi bước
nhưng có nhiễu → cần bước giảm dần ($\sum\eta_k=\infty$, $\sum\eta_k^2<\infty$).

**4.3 Subgradient** — $w_{k+1}=w_k-\eta_k g_k$ với $g_k\in\partial F(w_k)$.

**4.4 Proximal** — $w_{k+1}=\mathrm{prox}_{tg}\big(w_k-t\nabla f(w_k)\big)$.

Hai hàm cốt lõi của 4.3 và 4.4 được định nghĩa ngay dưới đây.

---
### HÀM DƯỚI ĐẠO HÀM (subgradient)

Khi có $\lambda_1\|w\|_1$, hàm mục tiêu **không khả vi tại $w_j=0$** — mà đó lại đúng là
chỗ nghiệm hay nằm. Thay gradient bằng **dưới vi phân**:

$$\partial F(w)=\nabla f(w)+\lambda_1\,\partial\|w\|_1,\qquad
\partial|w_j|=\begin{cases}\{\mathrm{sign}(w_j)\} & w_j\neq 0\\ [-1,1] & w_j=0\end{cases}$$

Tại $w_j=0$ dưới vi phân là **cả một đoạn** nên phải *chọn* một phần tử. Chọn phần tử có
**chuẩn nhỏ nhất**:

$$g_j=\nabla_j f(w)-\mathrm{clip}\big(\nabla_j f(w),\,-\lambda_1,\,\lambda_1\big)$$

vì $\|g\|=0$ **khi và chỉ khi** $0\in\partial F(w)$, tức $w$ tối ưu → dùng luôn làm tiêu
chuẩn dừng (điều kiện KKT).

In [ ]:
# ============ DINH NGHIA HAM DUOI DAO HAM ============
# (day la ban trong logloss_optim.py, chep lai de tien theo doi)
def duoi_dao_ham(w, X, y, lam1=0.0, lam2=0.0, idx=None, cach_chon="min_norm"):
    # idx: chi so mini-batch -> duoi dao ham ngau nhien
    if idx is not None:
        g = grad(w, X[idx], y[idx], lam2)
    else:
        g = grad(w, X, y, lam2)
    if lam1 == 0:
        return g

    g = g.copy()
    khac_0 = w != 0
    bang_0 = w == 0
    khac_0[0] = bang_0[0] = False          # bo intercept (khong bi phat)

    g[khac_0] += lam1 * np.sign(w[khac_0])            # w_j != 0: dao ham duy nhat
    if cach_chon == "min_norm":                       # w_j == 0: phai CHON
        g[bang_0] -= np.clip(g[bang_0], -lam1, lam1)  # phan tu chuan nho nhat
    elif cach_chon == "random":
        g[bang_0] += np.random.uniform(-lam1, lam1, bang_0.sum())
    return g

w0 = np.zeros(d)
print("khop voi ban trong module:",
      np.allclose(duoi_dao_ham(w0, X_tr, y_tr, lam1, lam2),
                  subgradient(w0, X_tr, y_tr, lam1, lam2)))
print("||g|| tai w = 0:", np.linalg.norm(subgradient(w0, X_tr, y_tr, lam1, lam2)))

In [ ]:
# Minh hoa: duoi vi phan cua |w| tai 0 la ca doan [-1, 1]
t = np.linspace(-1.5, 1.5, 400)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))

ax[0].plot(t, np.abs(t), color=MAU[0], lw=2)
for s in (-1, -0.5, 0, 0.5, 1):                    # cac tiep tuyen duoi tai 0
    ax[0].plot(t, s * t, color=MAU[1], lw=1, alpha=0.55)
ax[0].set_title("$|w|$ va cac duong tiep tuyen duoi tai 0", loc="left")

ax[1].plot(t[t < 0], -np.ones((t < 0).sum()), color=MAU[0], lw=2)
ax[1].plot(t[t > 0], np.ones((t > 0).sum()), color=MAU[0], lw=2)
ax[1].plot([0, 0], [-1, 1], color=MAU[1], lw=3)    # doan [-1,1] tai w=0
ax[1].scatter([0], [0], color=MAU[1], s=25, zorder=3)
ax[1].set_title(r"$\partial|w|$: da tri $[-1,1]$ tai $w=0$", loc="left")

for a in ax:
    a.axhline(0, color="#c3c2b7", lw=0.8)
    a.axvline(0, color="#c3c2b7", lw=0.8)
    a.grid(True, color="#e1e0d9", lw=0.8)
plt.tight_layout(); plt.show()

---
### TOÁN TỬ PROXIMAL

$$\mathrm{prox}_{tg}(v)=\arg\min_u\Big\{g(u)+\frac{1}{2t}\|u-v\|_2^2\Big\}$$

Với $g(w)=\lambda_1\|w\|_1$, bài toán **tách theo từng toạ độ** và có nghiệm dạng đóng —
**soft-thresholding**:

$$\big[\mathrm{prox}_{t\lambda_1\|\cdot\|_1}(v)\big]_j=\mathrm{sign}(v_j)\max\big(|v_j|-t\lambda_1,\,0\big)$$

Đây là điểm mấu chốt: toán tử này **đặt hẳn** các toạ độ nhỏ **về đúng 0** → nghiệm thưa
thật sự. Với Elastic-Net thì
$\mathrm{prox}_{tg}(v)=\dfrac{\mathrm{soft}(v,\,t\lambda_1)}{1+t\lambda_2}$.

In [ ]:
# ============ DINH NGHIA TOAN TU PROXIMAL ============
def prox_cua_L1(v, t, lam1):
    u = v.copy()
    u[1:] = np.sign(v[1:]) * np.maximum(np.abs(v[1:]) - t * lam1, 0)
    return u                                   # u[0]: intercept khong bi phat

def prox_elastic(v, t, lam1, lam2):
    u = prox_cua_L1(v, t, lam1)
    u[1:] = u[1:] / (1 + t * lam2)
    return u

# kiem chung bang cach giai truc tiep bai toan 1 chieu tren luoi
v, tt, l1 = np.array([0.0, 0.7, -0.30, 0.05, 2.0]), 0.5, 0.4
luoi = np.linspace(-4, 4, 400001)
giai_tay = [v[0]] + [luoi[np.argmin(l1 * np.abs(luoi) + (luoi - v[j]) ** 2 / (2 * tt))]
                     for j in range(1, 5)]
print("cong thuc dang dong :", prox_cua_L1(v, tt, l1))
print("giai truc tiep      :", np.array(giai_tay))
print("sai lech toi da     :", np.abs(prox_cua_L1(v, tt, l1) - np.array(giai_tay)).max())

In [ ]:
# Hinh dang cua soft-thresholding
v = np.linspace(-3, 3, 600)
fig, ax = plt.subplots(figsize=(5.4, 3.8))
ax.plot(v, v, color="#898781", lw=1, ls="--", label="dong nhat (khong phat)")
for i, nguong in enumerate([0.3, 0.8, 1.5]):
    ax.plot(v, np.sign(v) * np.maximum(np.abs(v) - nguong, 0), color=MAU[i], lw=2,
            label="$t\\lambda_1$ = %.1f" % nguong)
ax.axhline(0, color="#c3c2b7", lw=0.8); ax.axvline(0, color="#c3c2b7", lw=0.8)
ax.grid(True, color="#e1e0d9", lw=0.8)
ax.set_xlabel("v"); ax.set_ylabel("prox(v)")
ax.set_title("Soft-thresholding: vung phang = he so bi ep ve 0", loc="left")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Kiem tra nhat quan: lam1 = 0 thi prox la anh xa dong nhat -> ISTA phai trung GD
a = proximal_gradient(X_tr, y_tr, 0.0, lam2, buoc=eta, so_vong=30, kieu_prox="khong")
b = gradient_descent(X_tr, y_tr, lam2, buoc=eta, so_vong=30)
print("ISTA(prox=khong) == GD ?", np.allclose(a[0], b[0]),
      "| lech toi da:", np.abs(a[0] - b[0]).max())

---
## 5. VÒNG 1 — cả 4 thuật toán dùng CHUNG một bộ tham số

$$w_0=0 \quad|\quad \eta=1/L\ \text{(cố định)} \quad|\quad 60\ \text{lần duyệt dữ liệu}
\quad|\quad B=1024\ \text{(SGD)} \quad|\quad \lambda_1,\lambda_2\ \text{như nhau}$$

Không bật cải tiến nào: không line search, không momentum, không trung bình, không gia tốc.

> **Đọc bảng cho đúng:** GD/SGD chỉ tối ưu phần trơn $f$ (không xử lý được L1), còn
> Subgradient/Proximal tối ưu $F=f+\lambda_1\|w\|_1$. Cột log-loss là log-loss **trần**
> (không cộng phạt) nên so sánh chéo được giữa cả 4 thuật toán.

In [ ]:
kq1 = [
    ("buoc 1/L",
     gradient_descent(X_tr, y_tr, lam2, buoc=eta, so_vong=VONG, val=val)),
    ("buoc 1/L, B=%d" % B,
     sgd(X_tr, y_tr, lam2, batch=B, so_epoch=VONG, eta0=eta,
         kieu_buoc="hang_so", val=val)),
    ("buoc 1/L",
     subgradient_method(X_tr, y_tr, lam1, lam2, eta0=eta, so_vong=VONG,
                        kieu_buoc="hang_so", val=val)),
    ("ISTA, buoc 1/L",
     proximal_gradient(X_tr, y_tr, lam1, lam2, buoc=eta, so_vong=VONG, val=val)),
]

lam1_tung_tt = [0.0, 0.0, lam1, lam1]        # GD/SGD khong co L1
ev1 = [danh_gia(w, ls, X_tr, y_tr, X_va, y_va, l1, lam2)
       for (_, (w, ls, _)), l1 in zip(kq1, lam1_tung_tt)]

in_bang([(TEN_4[i], kq1[i][0], ev1[i]) for i in range(4)], "KET QUA VONG 1")

In [ ]:
# F* tham chieu: chay FISTA that lau cho tung bai toan
bo_nho_F = {}
def F_sao_cua(l1):
    if l1 not in bo_nho_F:
        w, _, _ = proximal_gradient(X_tr, y_tr, l1, lam2, so_vong=1500, gia_toc=True,
                                    kieu_prox="l1" if l1 > 0 else "khong")
        bo_nho_F[l1] = (objective(w, X_tr, y_tr, l1, lam2), w)
    return bo_nho_F[l1][0]

F_sao, F_sao_tron = F_sao_cua(lam1), F_sao_cua(0.0)
print("F* (lam1=%g) = %.10f | F* (chi phan tron) = %.10f" % (lam1, F_sao, F_sao_tron))

ve1 = [(TEN_4[i], kq1[i][1][1]) for i in range(4)]
ve_hoi_tu(ve1, [F_sao_tron, F_sao_tron, F_sao, F_sao],
          "figures/nb_01_vong1.png", "Vong 1 - chung tham so (buoc 1/L)")
ve_val(ve1, "figures/nb_01b_vong1_val.png", "Vong 1 - log-loss validation", moc)

## 6. VÒNG 2 — mỗi thuật toán dò tham số RIÊNG

Giữ nguyên ngân sách **60 lần duyệt dữ liệu** cho mọi cấu hình (nên vẫn công bằng);
chỉ tham số thay đổi. Chọn cấu hình tốt nhất theo **log-loss validation**.

| Thuật toán | Lưới tham số |
|---|---|
| GD | bước $c/L$, $c\in\{1,2,5,10,20,50,100,200\}$, và line search Armijo |
| SGD | $B\in\{256,1024,8192\}$ × {hằng số, $\eta_0/(1+\gamma k)$, $\eta_0/\sqrt k$}, thêm momentum / trung bình Polyak |
| Subgradient | $\eta_0\in\{1,5,20,50,200\}/L$ × 3 lịch bước × $\lambda_1\in\{10^{-3},10^{-4}\}$ |
| Proximal | {ISTA, FISTA} × {bước $1/L$, line search} × $\lambda_1\in\{10^{-3},10^{-4}\}$, + prox Elastic-Net |

$\lambda_1$ nằm trong lưới của Subgradient/Proximal vì nó đánh đổi trực tiếp giữa độ thưa
và độ chính xác — ép $\lambda_1$ cố định như vòng 1 là bắt hai thuật toán này chịu thiệt
trên log-loss. GD/SGD không có tham số này.

In [ ]:
luoi_gd = [("buoc %g/L" % c, {"buoc": eta * c}) for c in (1, 2, 5, 10, 20, 50, 100, 200)]
luoi_gd += [("line search Armijo", {"line_search": True})]

luoi_sgd = []
for b in (256, 1024, 8192):
    for kb in ("hang_so", "1/k", "1/sqrt(k)"):
        luoi_sgd.append(("B=%d, buoc %s" % (b, kb),
                         {"batch": b, "eta0": eta, "kieu_buoc": kb}))
for e0 in (0.5, 2.0):
    luoi_sgd.append(("B=%d, eta0=%g, buoc 1/k" % (B, e0),
                     {"batch": B, "eta0": e0, "kieu_buoc": "1/k"}))
luoi_sgd.append(("B=%d, momentum 0.9" % B,
                 {"batch": B, "eta0": eta, "kieu_buoc": "1/k", "momentum": 0.9}))
luoi_sgd.append(("B=%d, trung binh Polyak" % B,
                 {"batch": B, "eta0": eta, "kieu_buoc": "1/sqrt(k)", "trung_binh": True}))

luoi_sub, luoi_prox = [], []
for l1 in (lam1, lam1 / 10):
    for c in (1, 5, 20, 50, 200):
        for kb in ("hang_so", "1/sqrt(k)", "1/k"):
            luoi_sub.append(("eta0=%g/L, %s, lam1=%g" % (c, kb, l1),
                             {"eta0": eta * c, "kieu_buoc": kb, "lam1": l1}))
    for gt, ten_gt in ((False, "ISTA"), (True, "FISTA")):
        for lsr, ten_ls in ((False, "buoc 1/L"), (True, "line search")):
            luoi_prox.append(("%s, %s, lam1=%g" % (ten_gt, ten_ls, l1),
                              {"gia_toc": gt, "line_search": lsr, "lam1": l1}))
    luoi_prox.append(("FISTA, prox Elastic-Net, lam1=%g" % l1,
                      {"gia_toc": True, "kieu_prox": "elastic_net", "lam1": l1}))

print("so cau hinh:", len(luoi_gd), len(luoi_sgd), len(luoi_sub), len(luoi_prox))

In [ ]:
t0 = time.time()

tot_gd = do_tham_so("GD", luoi_gd,
                    lambda ts: gradient_descent(X_tr, y_tr, lam2, so_vong=VONG,
                                                val=val, **ts),
                    X_tr, y_tr, X_va, y_va, 0.0, lam2)
tot_sgd = do_tham_so("SGD", luoi_sgd,
                     lambda ts: sgd(X_tr, y_tr, lam2, so_epoch=VONG, val=val, **ts),
                     X_tr, y_tr, X_va, y_va, 0.0, lam2)
tot_sub = do_tham_so("Subgradient", luoi_sub,
                     lambda ts: subgradient_method(X_tr, y_tr, lam2=lam2,
                                                   so_vong=VONG, val=val, **ts),
                     X_tr, y_tr, X_va, y_va, None, lam2)
tot_prox = do_tham_so("Proximal", luoi_prox,
                      lambda ts: proximal_gradient(X_tr, y_tr, lam2=lam2,
                                                   so_vong=VONG, val=val, **ts),
                      X_tr, y_tr, X_va, y_va, None, lam2)

print("\n(tong thoi gian do tham so: %.1fs)" % (time.time() - t0))

In [ ]:
tot = [tot_gd, tot_sgd, tot_sub, tot_prox]
ev2 = [t[1] for t in tot]                    # moi phan tu: (cfg, chi_so, w, ls, tham_so)
in_bang([(TEN_4[i], tot[i][0], ev2[i]) for i in range(4)],
        "KET QUA VONG 2 (cau hinh tot nhat cua moi thuat toan)")

ve2 = [(TEN_4[i], tot[i][3]) for i in range(4)]
ve_hoi_tu(ve2, [F_sao_tron, F_sao_tron,
                F_sao_cua(tot_sub[4]["lam1"]), F_sao_cua(tot_prox[4]["lam1"])],
          "figures/nb_02_vong2.png", "Vong 2 - tham so rieng")
ve_val(ve2, "figures/nb_02b_vong2_val.png", "Vong 2 - log-loss validation", moc)
ve_do_thua(ve2, "figures/nb_03_do_thua.png", d, "Do thua cua nghiem sau vong 2")
ve_2_vong(ev1, ev2, "figures/nb_04_vong1_vs_vong2.png")

## 7. Nhận xét

In [ ]:
print("%-13s%12s%12s%11s   %s" %
      ("thuat toan", "val vong 1", "val vong 2", "cai thien", "cau hinh thang"))
print("-" * 96)
for i in range(4):
    ct = (ev1[i]["ll_va"] - ev2[i]["ll_va"]) / ev1[i]["ll_va"] * 100
    print("%-13s%12.6f%12.6f%10.2f%%   %s" %
          (TEN_4[i], ev1[i]["ll_va"], ev2[i]["ll_va"], ct, tot[i][0]))

print("\nDo thua o VONG 1 (cung lam1 = %g):" % lam1)
for i in range(4):
    print("  %-13s nnz = %d/%d" % (TEN_4[i], ev1[i]["nnz"], d))

# Kiem chung dieu kien toi uu KKT: 0 thuoc dF(w*)
print("\n||duoi dao ham chuan nho nhat||:")
for ten, wv in [("nghiem tham chieu", bo_nho_F[lam1][1]), ("w = 0", np.zeros(d))]:
    print("  tai %-20s = %.3e" %
          (ten, np.linalg.norm(subgradient(wv, X_tr, y_tr, lam1, lam2))))

**(a) "Cùng một bộ tham số" không có nghĩa là công bằng.** Bước $1/L$ suy ra từ chặn
Lipschitz, nhưng đó là bước *của GD*:

- với **SGD** nó quá nhỏ một cách vô lý, vì mỗi epoch SGD đi $n/|B|$ bước chứ không phải 1
  → SGD vẫn về đích sớm nhất dù dùng đúng bước đó;
- với **Subgradient** nó sai *kiểu*: lý thuyết đòi bước giảm dần
  ($\sum\eta_k=\infty,\ \sum\eta_k^2<\infty$), bước hằng chỉ đưa nghiệm tới một **lân cận**
  của $w^\star$ rồi dao động quanh đó;
- với **GD/ISTA** thì $1/L$ đúng nhưng rất bi quan (chặn cho trường hợp xấu nhất) —
  bước lớn hơn nhiều lần vẫn chạy tốt.

**(b) Sau khi tinh chỉnh riêng**, khoảng cách giữa cấu hình tốt nhất và tệ nhất *trong
cùng một thuật toán* lớn hơn nhiều so với khoảng cách *giữa các thuật toán* → chọn tham
số quan trọng hơn chọn thuật toán. Mỗi thuật toán "thích" một kiểu tham số riêng:
GD cần bước lớn hơn $1/L$ hoặc line search; SGD cần lịch giảm dần; Subgradient bắt buộc
giảm bước; Proximal chỉ cần bật gia tốc Nesterov (FISTA).

**(c) Về độ thưa** — khác biệt bản chất giữa Subgradient và Proximal:

| | cơ chế tạo số 0 |
|---|---|
| GD, SGD | không có |
| Subgradient | chỉ là *phụ phẩm* của quy ước chọn phần tử trong $\partial F$: khi $\|\nabla_j f\|\le\lambda_1$ thì phần tử chuẩn nhỏ nhất bằng đúng 0 nên $w_j$ "dính" lại 0. Đổi sang `cach_chon="random"` là **đặc hoàn toàn**. |
| Proximal | **theo cấu trúc**: soft-thresholding đặt hẳn hệ số về 0 ở *mọi* vòng lặp |

**(d) Tốc độ lý thuyết** — lý do proximal thắng khi có L1:

| | Bài toán | Tốc độ | Số vòng để đạt sai số $\varepsilon$ |
|---|---|---|---|
| GD | trơn | $O(1/k)$ | $O(1/\varepsilon)$ |
| SGD | trơn | $O(1/\sqrt k)$ | mỗi bước rẻ hơn $n/\|B\|$ lần |
| Subgradient | + L1 | $O(1/\sqrt k)$ | $O(1/\varepsilon^2)$ |
| ISTA | + L1 | $O(1/k)$ | $O(1/\varepsilon)$ |
| FISTA | + L1 | $O(1/k^2)$ | $O(1/\sqrt\varepsilon)$ |

**(e) Số lần duyệt dữ liệu ≠ thời gian thực.** SGD tiến nhanh nhất *tính theo số lần
duyệt dữ liệu*, nhưng lại **chậm hơn** tính theo giây: mỗi epoch nó gọi $n/|B|\approx 614$
phép nhân ma trận nhỏ, còn GD/ISTA chỉ gọi **một** phép nhân lớn trên cả 628k dòng —
numpy/BLAS chạy phép lớn hiệu quả hơn nhiều, nhất là khi $d=93$ khá nhỏ. Ưu thế lý
thuyết $O(|B|d)$ của SGD chỉ thành ưu thế thực tế khi $d$ lớn hoặc dữ liệu không vừa
bộ nhớ.

**Kết luận:** khi hàm mục tiêu tách được thành *trơn + không trơn*, đừng dùng subgradient
— hãy khai thác cấu trúc: gradient cho phần trơn, **prox** cho phần không trơn. Và luôn
so sánh trên **cùng một ngân sách**, đồng thời nhìn cả hai trục: số lần duyệt dữ liệu
*và* thời gian thực.

## 8. (Tuỳ chọn) Huấn luyện cả 24 sản phẩm → MAP@7

Metric chính thức của cuộc thi: với mỗi khách, xếp hạng 24 sản phẩm theo xác suất,
**loại các sản phẩm đã sở hữu ở tháng $t-1$** (không thể "thêm mới"), lấy 7 sản phẩm đầu.

In [ ]:
tham_so = dict(tot_prox[4])                  # cau hinh proximal thang o vong 2
cot_lag = [ten_cot.index(c + "_lag") for c in TARGET_COLS]
da_co = (X_va[:, cot_lag] > X_va[:, cot_lag].min(axis=0) + 1e-9).astype(int)

P = np.zeros((len(X_va), 24))
t0 = time.time()
for j, sp in enumerate(TARGET_COLS):
    yj = Y_tr[:, j].astype(float)
    if yj.sum() < 10:                        # qua hiem thi lay ti le nen
        P[:, j] = yj.mean()
        continue
    wj, _, _ = proximal_gradient(X_tr, yj, lam2=lam2, so_vong=VONG, **tham_so)
    P[:, j] = du_doan(X_va, wj)

ti_le = (Y_va.sum(axis=1) > 0).mean()
print("MAP@7 (validation 2016-05-28), %.1fs" % (time.time() - t0))
print("  tren khach CO them moi (%.2f%% so khach) = %.6f" %
      (ti_le * 100, map_at_7(P, Y_va, da_co, True)))
print("  tren TOAN BO khach (cach Kaggle cham)   = %.6f" %
      map_at_7(P, Y_va, da_co, False))